# Step 11 — genes and principal components, federated

**Data type: RNA_array** (GSE65391). **Reads:** `step10_site_{A,B,C}.rds`, `step04_genes.rds`.
**Writes:** `step11_features.rds`, `step11_site_{A,B,C}.rds`.

Clustering uses **discovery patients only, first visit each**: one sample per child, so no child
counts more than once, and validation patients stay unseen.

| choice | reason | federated how |
|---|---|---|
| the 1,000 most variable expressed genes | genes that do not vary cannot separate patients | variance per gene from each site's n, sum and sum of squares |
| centre genes, do not scale to unit variance | a strongly varying programme such as interferon keeps its weight | the pooled mean from the same sums |
| first 10 principal components | later components are mostly measurement noise | subspace iteration (below) |

**Principal components without a Gram matrix.** The usual route builds the 1,000 × 1,000 gene-by-gene
matrix, which a site must not send: with about 37 patients a site, it could be unpicked towards the
patients' data. Instead the coordinator holds a guess of the components, a 1,000 × 20 basis V. Each
site returns X′XV for its own centred patients, also 1,000 × 20. The sum over sites is the pooled
X′XV, which improves the guess. Repeat until it stops changing (`federated_pca` in
`src/federation.R`).

In [1]:
source("../src/paths.R")
source("../src/federation.R")
start_log("11")
genes <- readRDS(art("step04_genes.rds"))
expr  <- genes$keep[genes$expressed]
core_of <- function(m) {                          # first visit of each discovery patient
  d <- m[m$split == "discovery", ]; d <- d[order(d$subject, d$visit), ]
  rownames(d)[!duplicated(d$subject)]
}
X <- lapply(setNames(SITES, SITES), function(s) {
  d <- readRDS(site_file("10", s)); d$E[expr, core_of(d$meta)]
})
sapply(X, ncol)

A  B  C 
37 37 36

## The 1,000 genes

In [2]:
N_GENES <- 1000
ps  <- pool_suff(lapply(SITES, function(s) send(site_suff(X[[s]]), s, "per-gene sums, discovery", ncol(X[[s]]))))
top <- names(sort(ps$var, decreasing = TRUE))[seq_len(N_GENES)]
centre <- ps$mean[top]
c(genes = length(top), ifn_genes_among_them = sum(read_gene_set("ifn-type1-6.txt") %in% top))

genes ifn_genes_among_them 
                1000                    5

## Ten principal components

In [3]:
N_PCS <- 10
Xt <- lapply(X, function(x) x[top, ])
pc <- federated_pca(Xt, centre, ps$n, k = N_PCS)
c(iterations = pc$iterations)
round(pc$sdev^2 / sum(ps$var[top]), 3)          # share of variance per component

iterations 
        97

[1] 0.200 0.103 0.071 0.054 0.051 0.031 0.030 0.027 0.021 0.020

## Oracle — possible only because this is a simulation

The same genes and components computed on the pooled discovery patients.

In [4]:
pooled <- do.call(cbind, X)
central_top <- names(sort(apply(pooled, 1, var), decreasing = TRUE))[seq_len(N_GENES)]
cp <- prcomp(t(pooled[central_top, ]), rank. = N_PCS)
gate <- c(same_genes = identical(sort(central_top), sort(top)),
          max_sdev_gap = max(abs(cp$sdev[1:N_PCS] - pc$sdev)),
          min_abs_cosine = min(abs(diag(crossprod(cp$rotation[top, ], pc$rotation)))))
gate
stopifnot(gate["same_genes"] == 1, gate["max_sdev_gap"] < 1e-8, gate["min_abs_cosine"] > 1 - 1e-8)
cat("GATE PASSED: federated genes and components equal the central ones\n")
rm(pooled)

same_genes   max_sdev_gap min_abs_cosine 
  1.000000e+00   2.131628e-14   1.000000e+00

GATE PASSED: federated genes and components equal the central ones


## Each site projects its own patients

In [5]:
for (s in SITES) {
  sc <- t(Xt[[s]] - centre) %*% pc$rotation
  colnames(sc) <- paste0("PC", seq_len(N_PCS))
  saveRDS(list(core = colnames(Xt[[s]]), scores = sc), site_file("11", s))
}
saveRDS(list(genes = top, centre = centre, sd = sqrt(ps$var[top]), rotation = pc$rotation,
             sdev = pc$sdev, n = ps$n), art("step11_features.rds"))
cat("wrote", art("step11_features.rds"), "\n")

wrote /Users/adeslatt/Scitechcon Dropbox/Anne DeslattesMays/projects/endotypes-transcriptomics/data/run_artifacts/step11_features.rds 


## Findings

The federated gene list and components are the ones a central analysis would find, without any site
sending a gene-by-gene matrix or a patient's values.